# CodeAlpha Task 1 – Credit Scoring Model

This notebook builds and evaluates multiple classification models (**Logistic Regression**, **Decision Tree**, **Random Forest**) for predicting credit risk based on applicant demographic, financial, and credit history features.

## Cell 1 - Import Libraries

In [ ]:
# Import core libraries for data handling, visualization, modeling, and evaluation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import is_numeric_dtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
import warnings
warnings.filterwarnings("ignore")

## Cell 2 - Load Dataset

In [ ]:
# Load dataset and display first 5 rows
df = pd.read_csv("credit_data.csv")
df.head()

## Cell 3 - Dataset Information

In [ ]:
# Print shape, info, summary stats, and missing value counts
print("Shape :", df.shape)
df.info()
df.describe()
df.isnull().sum()

## Cell 4 - Check Missing Values

In [ ]:
# Plot heatmap of missing values
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values")
plt.show()

## Cell 5 - Feature Engineering

### Fill Missing Values & Encode Categorical Columns

In [ ]:
# Impute missing values (median for numeric, mode for categorical)
for column in df.columns:
    if is_numeric_dtype(df[column]):
        df[column] = df[column].fillna(df[column].median())
    else:
        df[column] = df[column].fillna(df[column].mode()[0])

# Encode categorical text columns to numeric integers
encoder = LabelEncoder()
for column in df.columns:
    if not is_numeric_dtype(df[column]):
        df[column] = encoder.fit_transform(df[column].astype(str))

## Cell 6 - Correlation Heatmap

In [ ]:
# Plot correlation matrix heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## Cell 7 - Define Features and Target

In [ ]:
# Separate input features (X) and target label (y)
X = df.drop("Credit_Status", axis=1)
y = df["Credit_Status"]

## Cell 8 - Train Test Split

In [ ]:
# Split data into 80% train and 20% test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Cell 9 - Feature Scaling

In [ ]:
# Standardize numerical features (mean=0, std=1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Logistic Regression

## Cell 10

In [ ]:
# Train Logistic Regression model and generate predictions
lr = LogisticRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

## Cell 11 - Evaluation

In [ ]:
# Evaluate Logistic Regression model
print("Accuracy :", accuracy_score(y_test, lr_pred))
print("Precision :", precision_score(y_test, lr_pred))
print("Recall :", recall_score(y_test, lr_pred))
print("F1 Score :", f1_score(y_test, lr_pred))
print("ROC AUC :", roc_auc_score(y_test, lr_prob))
print(classification_report(y_test, lr_pred))

# Decision Tree

## Cell 12

In [ ]:
# Train Decision Tree model and generate predictions
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_prob = dt.predict_proba(X_test)[:, 1]

## Cell 13

In [ ]:
# Evaluate Decision Tree model
print("Accuracy :", accuracy_score(y_test, dt_pred))
print("Precision :", precision_score(y_test, dt_pred))
print("Recall :", recall_score(y_test, dt_pred))
print("F1 Score :", f1_score(y_test, dt_pred))
print("ROC AUC :", roc_auc_score(y_test, dt_prob))
print(classification_report(y_test, dt_pred))

# Random Forest

## Cell 14

In [ ]:
# Train Random Forest model and generate predictions
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

## Cell 15

In [ ]:
# Evaluate Random Forest model
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision :", precision_score(y_test, rf_pred))
print("Recall :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))
print("ROC AUC :", roc_auc_score(y_test, rf_prob))
print(classification_report(y_test, rf_pred))

# Confusion Matrix

## Cell 16

In [ ]:
# Compute and plot Random Forest confusion matrix
cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

# ROC Curve

## Cell 17

In [ ]:
# Plot ROC Curve for Random Forest
fpr, tpr, _ = roc_curve(y_test, rf_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label="Random Forest")
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Compare Models

## Cell 18

In [ ]:
# Summarize evaluation metrics for all models and export to CSV
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Precision": [
        precision_score(y_test, lr_pred),
        precision_score(y_test, dt_pred),
        precision_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, lr_pred),
        recall_score(y_test, dt_pred),
        recall_score(y_test, rf_pred)
    ],
    "F1 Score": [
        f1_score(y_test, lr_pred),
        f1_score(y_test, dt_pred),
        f1_score(y_test, rf_pred)
    ],
    "ROC AUC": [
        roc_auc_score(y_test, lr_prob),
        roc_auc_score(y_test, dt_prob),
        roc_auc_score(y_test, rf_prob)
    ]
})
results.to_csv("model_accuracy_results.csv", index=False)
results

# Best Model

## Cell 19

In [ ]:
# Rank models by Accuracy in descending order
best_model = results.sort_values(by="Accuracy", ascending=False)
best_model

# Feature Importance

## Cell 20

In [ ]:
# Plot horizontal bar chart of feature importances
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values().plot(kind="barh", figsize=(8, 6))
plt.title("Feature Importance")
plt.show()

# Export Results

## Save Results to CSV

In [ ]:
# Save best model leaderboard and prediction results to CSV
best_model.to_csv("best_model_accuracy.csv", index=False)
predictions_df = pd.DataFrame({
    "Actual_Credit_Status": y_test,
    "Logistic_Regression_Pred": lr_pred,
    "Decision_Tree_Pred": dt_pred,
    "Random_Forest_Pred": rf_pred
})
predictions_df.to_csv("credit_predictions_results.csv", index=False)
print("Saved CSV files successfully!")

# Conclusion

## Cell 21 (Markdown)

### Conclusion

1. **Built a Credit Scoring Model** using Logistic Regression, Decision Tree, and Random Forest.
2. **Performed Preprocessing**: Missing value imputation, label encoding, and standardization.
3. **Evaluated Models**: Calculated Accuracy, Precision, Recall, F1 Score, and ROC-AUC.
4. **Visualized Results**: Confusion matrix, ROC curve, and feature importances.
5. **Exported CSVs**: Saved performance metrics and prediction results.